# Submission 1: Project Visualization (5%)

**Course:** RBB2013 / FFM2063 / FEM2063, Digital Twin, May 2026
**Project:** SmartClean Twin, a software-emulated Digital Twin of a mobile
inspection and cleaning robot (project topic 2)
**Repository:** https://github.com/KAI-UTP/smartclean-twin
**Presentation & demo video:** [https://youtu.be/zEq7L-ivMLA](https://youtu.be/zEq7L-ivMLA)

**Team Members**

| No | Name | Student ID |
|---|---|---|
| 1 | Chan Li Kai | 22010900 |
| 2 | William Wong Xiao Kang | 22010943 |
| 3 | Irvin Chang Hou Ceng | 22012342 |
| 4 | Liang Yan Ee | 22011522 |
| 5 | Nurin Emelin Binti Marhisyam | 24006706 |

> **How to reproduce:** start the stack with `docker compose up -d` (8 containers),
> then run the notebook top to bottom. All code cells in this notebook were
> executed against the running system and their outputs are saved below, so the
> evidence is readable without re-running.


## 1. Executive Summary

This submission documents the visualization layer of the SmartClean Twin. Two
independent visualizations are driven from a single time-series data store
(InfluxDB), which is the authoritative record of both raw sensor data and
derived Digital Twin state:

1. **Grafana operator dashboard**, 28 panels arranged in 7 collapsible
   sections in a control-room layout, auto-refreshing every 5 seconds. It
   presents live sensor time series, derived twin state, windowed
   aggregations, AI predictions and the alarm history.
2. **NVIDIA Omniverse 3D twin**, a USD scene in which the robot's pose,
   heading, cleaning coverage, battery level and safety state are updated once
   per second from the same InfluxDB source.

Because both read the same store, the dashboard and the 3D scene are always
consistent with each other, *one source of truth, two views*. Section 11
demonstrates this: a single injected fault changes both visualizations within
seconds.

The **measure of success** of the twin, cleaning coverage percentage, is
displayed in both: as a gauge and progress trend in Grafana, and as floor tiles
turning green in the 3D scene.


## 2. Purpose of the Digital Twin and the Role of Visualization

A cleaning robot normally operates unattended. A failing motor, a robot stuck
against an obstacle, or a battery that empties mid-room are all invisible to
the operator until the cleaning job is found to be incomplete. Manual
inspection does not scale to a fleet.

**SMART outcome of the twin.** The SmartClean Twin shall:

| SMART attribute | Statement |
|---|---|
| Specific | Monitor one cleaning robot (SCR01) in a 5 m × 5 m room, reporting pose, sensors, safety state and cleaning coverage |
| Measurable | Telemetry every 1 s; unsafe conditions surfaced within 5 s; cleaning coverage reported as a percentage of accessible cells |
| Achievable | Implemented with a simulated robot and eight containerised microservices on a single host |
| Relevant | Supports predictive maintenance and cleaning-completeness verification, the two operational questions an operator actually has |
| Time-bound | Delivered across two sprint cycles, completed July 2026 |

**Role of visualization.** The twin is only useful if a human can act on it.
Visualization is therefore not decoration: every panel exists to answer one of
four operator questions, *Is it safe? Is it healthy? Is it making progress?
What should I do?* Section 6 maps each dashboard section to one of these
questions.


## 3. Sensor Data versus Digital Twin State

Following the distinction made in Module 2, the visualization consumes **three
different kinds of information**, and it is important that they are not
conflated:

| Kind | Definition | Examples in this project | Stored as |
|---|---|---|---|
| **Sensor data** | Raw physical measurements reported by the asset | `motor_temperature_c`, `battery_soc`, `obstacle_cm`, `x_m`, `y_m` | measurement `robot_telemetry` |
| **Digital Twin state** | Interpreted, operator-level condition derived from sensor data by rules | `safety_state` = SAFE / WARNING / EMERGENCY, `battery_state`, `mission_state`, `cleaning_coverage_pct` | measurement `robot_state` |
| **Derived predictions** | Model outputs about conditions that are not directly measurable | `health_state`, `predicted_rul_minutes`, `anomaly_score`, `recommendation` | measurement `robot_prediction` |

A raw temperature reading of 78 °C is a *sensor value*; the conclusion
"motor OVERHEATED, safety state WARNING" is *twin state*; "approximately 30
minutes of useful life remain" is a *prediction*. The dashboard deliberately
shows all three, because an operator needs the interpretation, not only the
number, and needs the raw number available to verify the interpretation.


## 4. Data Source and Update Period Justification

### 4.1 Time-series store

Both visualizations query **InfluxDB 2.7** over HTTP using the Flux query
language. InfluxDB is used rather than a flat file or a relational table
because the access pattern is almost entirely *"latest value"* and
*"values over a time range, aggregated"*, which a time-series database answers
natively and cheaply.

In Grafana the datasource is **provisioned as code**
(`grafana/provisioning/datasources/influxdb.yaml`) and the dashboard itself is
provisioned from `grafana/dashboards/smartclean_twin.json`, both baked into the
Grafana container image. The dashboard is therefore reproducible on any machine
and is version-controlled, it is not hand-built through the UI.

### 4.2 Chosen update periods

| Stage | Period | Justification |
|---|---|---|
| Robot telemetry publish | 1 s | Fastest meaningful rate of change: the robot travels at ≤ 0.2 m/s, so 1 s ≈ 0.2 m of movement, fine enough to see motion and to catch an obstacle event before the robot advances a full cell (0.5 m) |
| Twin state evaluation | per telemetry message (1 s) | State must never lag the data it is derived from |
| Grafana dashboard refresh | 5 s | A human reads a dashboard on the order of seconds; 5 s keeps the display current while reducing query load fivefold versus 1 s. Worst-case display lag (1 s publish + 5 s refresh) stays inside the 5-second detection requirement in practice, and alarm panels reflect the state at the next tick |
| Omniverse 3D update | 1 s | Motion must appear continuous, so the 3D scene is updated at the full telemetry rate rather than the dashboard rate |

Sampling faster than 1 s would add network and storage cost without adding
information, because the simulator's physics tick is itself 1 s; sampling
slower than 1 s would risk missing a cell transition.


## 5. Visualization Architecture

```
   robot_telemetry ┐
   robot_state     ├─► InfluxDB :8086 ─┬─► Grafana :3001  (Flux, every 5 s)
   robot_prediction┘   (single source  │     28 panels / 7 sections
   robot_alert         of truth)       │
                                       └─► NVIDIA Omniverse (Flux, every 1 s)
                                             USD scene, 3D properties
```

Neither visualization talks to the robot or to any microservice directly. Both
are pure consumers of the store. Three consequences follow, and all three are
desirable:

1. **Consistency**, the dashboard and the 3D scene cannot disagree.
2. **Isolation**, a visualization crash cannot affect data collection.
3. **Replayability**, because the store is historical, both views can be
   pointed at a past time window to review an incident.


## 6. Grafana Dashboard Design, 28 panels in 7 sections

The dashboard is laid out as a control room: the top strip answers
*"is anything wrong right now?"* in one glance, and detail sections below
answer *"why?"*.

| # | Section | Panels | Operator question answered |
|---|---|---|---|
| 1 | Robot Status Overview | Safety State, Mission State, AI Health State, Anomaly Detected, AI Recommendation | Is it safe? What should I do? |
| 2 | Battery & Power | Battery SoC gauge, Battery Voltage, Battery Discharge Rate (derivative) | Will it finish before the battery empties? |
| 3 | Motion & Environment | Robot Position X, Obstacle Distance | Is it moving and unobstructed? |
| 4 | Motor & Cleaning | Motor Current, Motor Temperature, **Cleaning Coverage gauge**, **Cleaning Coverage Progress**, Dirt Score | Is the equipment healthy? Is it making progress? |
| 5 | AI Predictions & Forecasts | Motor Health, Dirt Level, Predicted RUL gauge, Predicted RUL trend, Anomaly Score, Minutes Until Return, Minutes to Finish | When will it need attention? |
| 6 | Statistical Trends (windowed) | Mean Motor Current (30 s), Max Motor Temperature (30 s), Mean Battery SoC (1 min) | What is the trend, ignoring noise? |
| 7 | Alarms & Events | Active Alarm Count, Alarm Count per Minute, Alarm History table | What has gone wrong, and how often? |

Panel types are chosen to match the data: **gauges** for bounded quantities with
thresholds (SoC, coverage, RUL), **time series** for anything whose trend
matters, **stat panels with value mapping** for categorical twin state
(SAFE/WARNING/EMERGENCY mapped to green/orange/red), and a **table** for the
discrete alarm event log.


## 7. Time-Series Visualizations

Time series are used where the *shape over time* carries the meaning rather
than the instantaneous value. Representative examples:

- **Motor Temperature (°C)**, a slow ramp indicates accumulating thermal load;
  a step indicates a fault. Instantaneous value alone cannot distinguish these.
- **Obstacle Distance (cm)**, a sharp drop is the signature of an obstacle
  event; the panel makes the approach visible before the emergency triggers.
- **Battery Discharge Rate (%/min)**, computed with the Flux `derivative()`
  function. Negative values indicate discharge, positive values indicate
  charging at the dock, so the autonomous charge cycle is directly visible.
- **Predicted RUL trend (minutes)**, shows the AI estimate declining as the
  robot works, which is more informative to a maintenance planner than a single
  current figure.


## 8. Aggregations and Transformations

Raw 1 Hz data is noisy. The dashboard therefore also presents server-side
aggregations, computed in Flux inside InfluxDB rather than in the browser, so
that only the reduced series is transferred:

| Aggregation | Flux operation | Purpose |
|---|---|---|
| Mean motor current over 30 s | `aggregateWindow(every: 30s, fn: mean)` | Smooths per-tick variation to reveal sustained load |
| Max motor temperature over 30 s | `aggregateWindow(every: 30s, fn: max)` | Worst case in the window, appropriate for a thermal safety limit, where the peak matters, not the average |
| Mean battery SoC over 1 min | `aggregateWindow(every: 1m, fn: mean)` | Long-horizon depletion trend |
| Battery discharge rate | `derivative(unit: 1m, nonNegative: false)` | Transformation from level to rate of change |
| Alarm count per minute | `aggregateWindow(every: 1m, fn: count)` | Event frequency rather than event detail |
| Coverage progress | `aggregateWindow(every: 10s, fn: last)` | Monotone progress series; `last` is correct here because coverage is cumulative, so averaging would understate it |

The choice of reducer is deliberate in each case: `mean` for load, `max` for a
safety limit, `last` for a cumulative counter, `count` for events. Section 9 of
Submission 3 shows these queries executing live.


## 9. NVIDIA Omniverse 3D Twin

### 9.1 Scene hierarchy

The scene is authored once by `omniverse/create_scene.py` and saved as USD:

```
/World
├── Room
│   ├── Floor                     5 m × 5 m
│   ├── WallNorth / South / East / West
│   ├── Desk01 / Desk02 / Desk03  obstacles at the grid's blocked cells
│   └── CeilingLight
├── CoverageGrid
│   └── Tile_0_0 … Tile_9_9        100 tiles, one per grid cell
├── CleaningRobot                  (parent Xform, moved and rotated)
│   ├── BrushDeck
│   ├── Body                       colour = safety state
│   ├── StatusLight                flashes during EMERGENCY
│   ├── BatteryBar                 X-scale = battery SoC
│   └── DirectionArrow             child prim, so it rotates with the parent
├── ObstacleIndicator              parked at z = −5 when inactive
└── Trail
    └── Dot_0 … Dot_59             breadcrumb path
```

The robot is a **hierarchical object**: `DirectionArrow`, `Body`,
`StatusLight`, `BatteryBar` and `BrushDeck` are children of the
`CleaningRobot` Xform, so translating and rotating the parent moves the whole
assembly and the arrow always points along the robot's heading without any
separate update.

### 9.2 Source value → 3D property mapping

`omniverse/live_update.py` polls InfluxDB every second and applies each source
value to exactly one 3D property:

| Source (InfluxDB) | 3D property updated | Prim |
|---|---|---|
| `robot_telemetry.x_m`, `y_m` | `translate` X, Y | `/World/CleaningRobot` |
| `robot_telemetry.heading_deg` | `rotate` Z | `/World/CleaningRobot` |
| `robot_telemetry.battery_soc` | `scale` X and `displayColor` | `/World/CleaningRobot/BatteryBar` |
| `robot_state.safety_state` | `displayColor` | `/World/CleaningRobot/Body` |
| `robot_state.safety_state` (EMERGENCY) | `displayColor`, toggled each tick | `/World/CleaningRobot/StatusLight` |
| `robot_state.safety_state` (EMERGENCY) | `translate` (surfaces in front of robot) | `/World/ObstacleIndicator` |
| derived: visited cell from `x_m`, `y_m` | `displayColor` → green | `/World/CoverageGrid/Tile_X_Y` |
| derived: successive positions | `translate` of the next dot | `/World/Trail/Dot_N` |

### 9.3 Verification that the correct value drives the correct property

Three checks were used, and each caught a real defect during development:

1. **Coordinate check.** `x_m` must map to the tile index as
   `int(x_m / CELL_SIZE_M)` with `CELL_SIZE_M = 0.5`. An initial version used
   `int(x_m)`, which lit tile 1 instead of tile 3 for x = 1.5 m. Fixed and
   verified by walking the robot to a known cell.
2. **Field-arrival check.** InfluxDB returns one table per field; the query is
   grouped and pivoted so that every requested field reaches the scene in one
   row. Before this fix `safety_state` frequently did not arrive and the robot
   stayed green during an EMERGENCY, a silent visualization error.
3. **Per-tick console trace.** Every tick prints the values actually applied,
   e.g. `[TICK] pos=(3.5,2.0) heading=180° safety=EMERGENCY coverage=30.5%
   battery=98.0% tiles_visited=37/100`, so the 3D result can be compared
   against the source numbers frame by frame.


## 10. Live evidence, the data consumed by both visualizations

The cells below query InfluxDB directly, showing the exact values that the
dashboard panels and the 3D scene were rendering at execution time.

In [1]:
import json, time, urllib.request

INFLUX = "http://localhost:8086/api/v2/query?org=smartclean"
TOKEN = "smartclean-super-secret-token"

def flux_query(q):
    """Run a Flux query against InfluxDB and return raw CSV."""
    req = urllib.request.Request(INFLUX, data=q.encode(),
        headers={"Authorization": f"Token {TOKEN}",
                 "Content-Type": "application/vnd.flux", "Accept": "application/csv"})
    with urllib.request.urlopen(req, timeout=10) as r:
        return r.read().decode()

def show_last(measurement, range_s=30):
    """Print the most recent value of every field in a measurement."""
    q = (f'from(bucket: "smartclean_twin") |> range(start: -{range_s}s) '
         f'|> filter(fn: (r) => r._measurement == "{measurement}") |> last()')
    n = 0
    for line in flux_query(q).splitlines():
        p = line.split(",")
        if len(p) > 7 and p[1] == "_result":
            print(f"  {p[7]:28s} = {p[6]}")
            n += 1
    if n == 0:
        print("  (no data in window, is the stack running?)")

print("Helper functions loaded.")


Helper functions loaded.


In [2]:
print("robot_telemetry (raw sensor data (drives sensor panels + 3D pose):")
show_last("robot_telemetry")
print()
print("robot_state (derived Digital Twin state (drives status strip + 3D colours):")
show_last("robot_state")
print()
print("robot_prediction (model outputs (drives AI Predictions section):")
show_last("robot_prediction")


robot_telemetry (raw sensor data (drives sensor panels + 3D pose):
  battery_a                    = 1.5
  battery_soc                  = 99.86
  battery_v                    = 12.596
  brush_on                     = 1
  bumper_active                = 0
  dirt_score                   = 0
  heading_deg                  = 0
  motor_current_a              = 0.9
  motor_temperature_c          = 26.5
  obstacle_cm                  = 20
  pump_on                      = 0
  sequence                     = 119
  speed_mps                    = 0
  water_level_pct              = 100
  x_m                          = 1
  y_m                          = 3.5

robot_state (derived Digital Twin state (drives status strip + 3D colours):
  alarm_count                  = 1
  battery_state                = NORMAL
  cleaning_coverage_pct        = 50.85
  connection_state             = ONLINE
  dirt_level                   = CLEAN
  mission_state                = PAUSED
  motion_state                 = STOPPED

## 11. Demonstration, a single fault changes both visualizations

An obstacle fault is injected into the robot through its fault-injection API.
The state engine raises `safety_state = EMERGENCY`, which the dashboard shows as
a red status strip and the 3D scene shows as a red robot body with a flashing
status light and the obstacle indicator surfacing in front of the robot. The
fault is then cleared and both return to green.

This is the strongest available evidence that the visualizations are genuinely
driven by twin state rather than displaying static content.

In [3]:
def inject(fault):
    req = urllib.request.Request("http://localhost:8004/fault",
        data=json.dumps({"fault": fault}).encode(),
        headers={"Content-Type": "application/json"}, method="POST")
    with urllib.request.urlopen(req, timeout=5) as r:
        print(f"POST /fault {fault!r} -> HTTP {r.status}")

print("BEFORE: twin state:")
show_last("robot_state", 10)

inject("obstacle")
time.sleep(15)
print()
print("DURING FAULT: twin state (dashboard strip red, 3D robot red):")
show_last("robot_state", 10)

inject("clear")
time.sleep(12)
print()
print("AFTER CLEARING: twin state restored:")
show_last("robot_state", 10)


BEFORE: twin state:
  alarm_count                  = 1
  battery_state                = NORMAL
  cleaning_coverage_pct        = 50.85
  connection_state             = ONLINE
  dirt_level                   = CLEAN
  mission_state                = PAUSED
  motion_state                 = STOPPED
  motor_health                 = NORMAL
  safety_state                 = EMERGENCY
  twin_quality                 = SYNCHRONIZED
POST /fault 'obstacle' -> HTTP 200



DURING FAULT: twin state (dashboard strip red, 3D robot red):
  alarm_count                  = 1
  battery_state                = NORMAL
  cleaning_coverage_pct        = 50.85
  connection_state             = ONLINE
  dirt_level                   = CLEAN
  mission_state                = PAUSED
  motion_state                 = STOPPED
  motor_health                 = NORMAL
  safety_state                 = EMERGENCY
  twin_quality                 = SYNCHRONIZED
POST /fault 'clear' -> HTTP 200



AFTER CLEARING: twin state restored:
  alarm_count                  = 1
  battery_state                = NORMAL
  cleaning_coverage_pct        = 50.85
  connection_state             = ONLINE
  dirt_level                   = CLEAN
  mission_state                = PAUSED
  motion_state                 = STOPPED
  motor_health                 = NORMAL
  safety_state                 = EMERGENCY
  twin_quality                 = SYNCHRONIZED


## 12. Discussion

The visualization layer meets the purpose set out in Section 2. In particular:

- **Alignment with purpose.** Every panel maps to one of the four operator
  questions (Section 6). There are no panels that display data simply because
  it is available.
- **Measure of success is visible.** Cleaning coverage appears as a gauge, as a
  progress trend, and as green tiles in 3D, three representations of the same
  quantity for three different reading distances.
- **Interpretation as well as measurement.** Raw sensor series are shown
  alongside the derived twin state and the AI recommendation, so the operator
  sees both the conclusion and the evidence for it.
- **Two synchronised views.** The dashboard is better for numbers and history;
  the 3D scene is better for spatial understanding, where the robot is, which
  part of the room is still dirty, and what it is about to collide with.

### 12.1 Limitations

1. The 3D scene must be started manually inside the Omniverse Script Editor;
   it is not a service in the compose stack.
2. Grafana has no authentication beyond the default admin account and no TLS;
   acceptable for a single-host prototype, not for deployment.
3. The 3D geometry is primitive (cylinders and cubes) rather than a CAD model
   of a real robot, because no physical asset exists yet.
4. Dashboard panels are fixed to robot `SCR01`; a fleet view would require
   templating panels on a `robot_id` variable.


## 13. Conclusion

Two complementary visualizations are driven at 5 s and 1 s respectively from a
single time-series store that holds raw sensor data, derived Digital Twin state
and model predictions as distinct measurements. The dashboard provides
28 purpose-mapped panels including time series, six windowed aggregations and
transformations, and the twin's measure of success; the Omniverse scene provides
a hierarchical 3D twin whose pose, colour, battery bar and coverage tiles are
each driven by a specified source field. A live fault injection demonstrates
both views reacting to the same state change within seconds.

Supporting evidence: dashboard capture and 3D capture appended to this
submission, and the demonstration video linked at the top of this notebook.
